# Step 2–3: Screening Eligibility, Test Assignment & Results

This notebook tests and demonstrates the screening layer of the simulation.

- **Step 2 — Eligibility**: `get_eligible_screenings()` checks each patient against USPSTF criteria for cervical and lung cancer. `days_until_eligible()` handles the three-way split: already eligible, not yet eligible (schedule return), or permanently ineligible (exit silently).
- **Step 3 — Test assignment & result draws**: `assign_screening_test()` picks the modality (age-stratified for cervical). `draw_cervical_result()` draws from a multinomial with risk-factor inflation for HPV+ / prior CIN history. `draw_lung_rads_result()` draws a Lung-RADS v2022 category.

All logic lives in `src/screening.py`. This notebook demonstrates and validates it in isolation.

**Handoff**: patients arrive here after being seen by a provider (runner.py queue layer). Results feed into `03_results_followup.ipynb`.

In [ ]:
import sys, random
sys.path.insert(0, '../src')   # ensure local .py files are importable

import config as cfg
from patient import Patient
from population import sample_patient
from screening import (
    get_eligible_screenings,
    days_until_eligible,
    is_due_for_screening,
    get_cervical_age_stratum,
    assign_screening_test,
    draw_cervical_result,
    draw_lung_rads_result,
    run_screening_step,
)

random.seed(cfg.RANDOM_SEED)
print('Imports OK')

## Eligibility Checks
Each cancer type has an eligibility function in `screening.py`.
`get_eligible_screenings(patient)` returns the full list for one patient.

In [2]:
# Edge-case patients to verify eligibility logic
cases = [
    # (label, age, has_cervix, smoker, pack_years, bmi)
    ('Age 25, with cervix, non-smoker', 25, True,  False, 0,  25),
    ('Age 45, with cervix, smoker 25pk', 45, True,  True,  25, 25),
    ('Age 45, no cervix',               45, False, False, 0,  25),
    ('Age 55, smoker 25pk',             55, True,  True,  25, 25),
    ('Age 68, BMI 17',                  68, True,  False, 0,  17),
    ('Age 70, all eligible',            70, True,  True,  25, 25),
    ('Age 20, below all thresholds',    20, True,  False, 0,  25),
]

print(f'{"Label":<35}  {"Eligible for"}' )
print('-' * 70)
for label, age, has_cervix, smoker, pack_years, bmi in cases:
    p = sample_patient(0, 0, 'pcp', 'outpatient')
    p.age, p.has_cervix, p.smoker = age, has_cervix, smoker
    p.pack_years, p.bmi = pack_years, bmi
    eligible = get_eligible_screenings(p)
    print(f'{label:<35}  {", ".join(eligible) if eligible else "(none)"}')

Label                                Eligible for
----------------------------------------------------------------------
Age 25, with cervix, non-smoker      cervical
Age 45, with cervix, smoker 25pk     cervical
Age 45, no cervix                    (none)
Age 55, smoker 25pk                  cervical, lung
Age 68, BMI 17                       (none)
Age 70, all eligible                 lung
Age 20, below all thresholds         (none)


## Test Assignment
Cervical test is age-stratified (ASCCP guidelines). Others use a single default.

In [3]:
ages = [22, 35, 50, 66]
print(f'{"Age":<6}  {"Stratum":<10}  {"Assigned test"}')
print('-' * 40)
for age in ages:
    p = sample_patient(0, 0, 'gynecologist', 'outpatient')
    p.age, p.has_cervix = age, True
    stratum = get_cervical_age_stratum(age)
    test    = assign_screening_test(p, 'cervical')
    print(f'{age:<6}  {stratum:<10}  {test}')

Age     Stratum     Assigned test
----------------------------------------
22      young       cytology
35      middle      hpv_alone
50      middle      hpv_alone
66      older       ineligible


## Result Distribution Check
Draw 5 000 results for a middle-aged patient and verify the distribution
matches the config probabilities (sanity check).

In [ ]:
from collections import Counter

N = 5_000
p_template = sample_patient(0, 0, 'gynecologist', 'outpatient')
p_template.age, p_template.has_cervix = 42, True
p_template.hpv_positive, p_template.prior_cin = False, None

# ── Cytology (age 21–65) ──────────────────────────────────────────────────────
cytology_results = [draw_cervical_result(p_template, 'cytology') for _ in range(N)]
counts = Counter(cytology_results)

print(f'Cytology result distribution (n={N:,}, age=42):')
for cat in ['NORMAL', 'ASCUS', 'LSIL', 'ASC-H', 'HSIL']:
    cnt      = counts.get(cat, 0)
    expected = cfg.CERVICAL_RESULT_PROBS['middle_cytology'].get(cat, 0)
    print(f'  {cat:<10} observed={cnt/N*100:5.1f}%  expected={expected*100:5.1f}%')

# ── HPV-alone (age 30–65) ─────────────────────────────────────────────────────
print()
hpv_results = [draw_cervical_result(p_template, 'hpv_alone') for _ in range(N)]
counts_hpv  = Counter(hpv_results)

print(f'HPV-alone result distribution (n={N:,}, age=42):')
for cat in ['HPV_NEGATIVE', 'HPV_POSITIVE']:
    cnt      = counts_hpv.get(cat, 0)
    expected = cfg.CERVICAL_RESULT_PROBS['middle_hpv'].get(cat, 0)
    print(f'  {cat:<15} observed={cnt/N*100:5.1f}%  expected={expected*100:5.1f}%')

## Smoke Test — Run 30 Patients Through Screening
Simulates one provider-seen day: generate patients, run `run_screening_step`
for each eligible cancer, collect results.

In [ ]:
from collections import defaultdict
from metrics import initialize_metrics

random.seed(42)
DAY        = 0
N_PATIENTS = 30

patients      = [sample_patient(i, DAY, 'gynecologist', 'outpatient') for i in range(N_PATIENTS)]
metrics       = initialize_metrics()
screening_log = defaultdict(list)

for p in patients:
    metrics['n_patients'] += 1
    eligible = get_eligible_screenings(p)

    if not eligible:
        # Three-way split (mirrors runner logic):
        #   not yet eligible → schedule return visit
        #   permanently ineligible → exit silently
        soonest = -1
        for cancer in cfg.ACTIVE_CANCERS:
            d = days_until_eligible(p, cancer)
            if d > 0 and (soonest < 0 or d < soonest):
                soonest = d
        if soonest > 0:
            screening_log['not_yet_eligible'].append(f'return in {soonest}d')
        else:
            screening_log['permanently_ineligible'].append('exit')
        metrics['n_unscreened'] += 1
        continue

    metrics['n_eligible_any'] += 1
    for cancer in eligible:
        result = run_screening_step(p, cancer, DAY, metrics)
        if result:
            screening_log[cancer].append(result)

print(f'Results across {N_PATIENTS} patients:\n')
for key, results in sorted(screening_log.items()):
    cnt = Counter(results)
    print(f'  {key}:')
    for r, c in sorted(cnt.items()):
        print(f'    {r:<30} {c}')

print(f'\n  Total seen:               {metrics["n_patients"]}')
print(f'  Eligible for any screen:  {metrics["n_eligible_any"]}')
print(f'  Not screened:             {metrics["n_unscreened"]}')

print('\n── Sample patient trace ──')
patients[0].print_history()